<a href="https://colab.research.google.com/github/iagooleite/Atividade1-POO/blob/main/atividade_sistemas_lineares.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Cálculo Numérico — Sistemas lineares e implementação numérica**
Iago da Silva Leite | RA: 185228


---






## Parte 1 — Conceitos e resolução manual

### Exercício 1 — Forma matricial $A\mathbf{x}=\mathbf{b}$

**a)**
$$A=\begin{bmatrix}2&1\\1&-3\end{bmatrix},\quad \mathbf{x}=\begin{bmatrix}x\\y\end{bmatrix},\quad \mathbf{b}=\begin{bmatrix}7\\-5\end{bmatrix}$$

Solução (complemento): $x = 16/7$, $y = 17/7$.

**b)**
$$A=\begin{bmatrix}1&2&-1\\3&-1&2\\2&1&1\end{bmatrix},\quad \mathbf{x}=\begin{bmatrix}x\\y\\z\end{bmatrix},\quad \mathbf{b}=\begin{bmatrix}2\\9\\7\end{bmatrix}$$

Solução (complemento): $x=2$, $y=1$, $z=2$.

In [1]:
# Verificação do Exercício 1 (complemento)
import numpy as np

A1a = np.array([[2, 1], [1, -3]], dtype=float)
b1a = np.array([7, -5], dtype=float)
print("1a:", np.linalg.solve(A1a, b1a), "-> esperado [16/7, 17/7] =", [16/7, 17/7])

A1b = np.array([[1, 2, -1], [3, -1, 2], [2, 1, 1]], dtype=float)
b1b = np.array([2, 9, 7], dtype=float)
print("1b:", np.linalg.solve(A1b, b1b), "-> esperado [2, 1, 2]")

1a: [2.28571429 2.42857143] -> esperado [16/7, 17/7] = [2.2857142857142856, 2.4285714285714284]
1b: [2. 1. 2.] -> esperado [2, 1, 2]


### Exercício 2 — Eliminação de Gauss

Matriz aumentada inicial:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\-3&-1&2&-11\\-2&1&2&-3\end{array}\right]$$

**Etapa 1.** Pivô $=2$. Operações: $L_2\leftarrow L_2+\tfrac32 L_1$ e $L_3\leftarrow L_3+L_1$:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&1/2&1/2&1\\0&2&1&5\end{array}\right]$$

**Etapa 2.** Pivô $=1/2$. Operação: $L_3\leftarrow L_3-4L_2$:
$$\left[\begin{array}{ccc|c}2&1&-1&8\\0&1/2&1/2&1\\0&0&-1&1\end{array}\right]$$

**Substituição regressiva:**
- $-z=1 \Rightarrow z=-1$
- $\tfrac12 y+\tfrac12(-1)=1 \Rightarrow y=3$
- $2x+3-(-1)=8 \Rightarrow x=2$

**Solução:** $(x,y,z)=(2,3,-1)$.  Verificação na 3ª equação: $-4+3-2=-3$ ✓

## Parte 2 — Propriedades e interpretação

### Exercício 3 — Número de soluções

**a)**
$$\left[\begin{array}{cc|c}1&1&3\\2&2&6\end{array}\right]\xrightarrow{L_2\leftarrow L_2-2L_1}\left[\begin{array}{cc|c}1&1&3\\0&0&0\end{array}\right]$$
Linha nula: posto $=1<2$ incógnitas e sistema consistente → **infinitas soluções** ($x=3-y$, $y$ livre).

**b)**
$$\left[\begin{array}{cc|c}1&1&3\\2&2&7\end{array}\right]\xrightarrow{L_2\leftarrow L_2-2L_1}\left[\begin{array}{cc|c}1&1&3\\0&0&1\end{array}\right]$$
A última linha equivale a $0=1$ (contradição) → **nenhuma solução**.

**c)**
$$\left[\begin{array}{cc|c}1&1&3\\2&-1&0\end{array}\right]\xrightarrow{L_2\leftarrow L_2-2L_1}\left[\begin{array}{cc|c}1&1&3\\0&-3&-6\end{array}\right]$$
Dois pivôs: posto $=2=$ número de incógnitas → **solução única**: $y=2$, $x=1$.

In [2]:
# Verificação do Exercício 3 usando o posto (rank) da matriz e da matriz aumentada
import numpy as np

def classificar(A, b):
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float).reshape(-1, 1)
    rA = np.linalg.matrix_rank(A)
    rAb = np.linalg.matrix_rank(np.hstack((A, b)))
    n = A.shape[1]
    if rA < rAb:
        return f"posto(A)={rA}, posto([A|b])={rAb} -> NENHUMA solução"
    if rA == n:
        return f"posto(A)={rA}, posto([A|b])={rAb} -> solução ÚNICA"
    return f"posto(A)={rA}, posto([A|b])={rAb} -> INFINITAS soluções"

print("3a:", classificar([[1, 1], [2, 2]], [3, 6]))
print("3b:", classificar([[1, 1], [2, 2]], [3, 7]))
print("3c:", classificar([[1, 1], [2, -1]], [3, 0]))

3a: posto(A)=1, posto([A|b])=1 -> INFINITAS soluções
3b: posto(A)=1, posto([A|b])=2 -> NENHUMA solução
3c: posto(A)=2, posto([A|b])=2 -> solução ÚNICA


## Parte 3 — Implementação numérica em Python

### Exercício 4 — Eliminação de Gauss sem pivoteamento

O programa: recebe $A$ e $\mathbf{b}$, forma a matriz aumentada, elimina os elementos abaixo dos pivôs, aplica substituição regressiva e exibe a solução. Em seguida, testa com o sistema do exercício 2 e compara com `numpy.linalg.solve`.

In [3]:
import numpy as np

def gauss_sem_pivoteamento(A, b):
    # Resolve Ax = b por eliminação de Gauss SEM pivoteamento.
    A = np.array(A, dtype=float)
    b = np.array(b, dtype=float)
    n = len(b)

    # 1) Matriz aumentada [A | b]
    M = np.hstack((A, b.reshape(-1, 1)))

    # 2) Eliminação: zera os elementos abaixo de cada pivô
    for k in range(n - 1):
        if M[k, k] == 0:
            raise ZeroDivisionError(f"Pivô nulo na posição ({k},{k}): é necessário pivoteamento.")
        for i in range(k + 1, n):
            m = M[i, k] / M[k, k]            # multiplicador
            M[i, k:] = M[i, k:] - m * M[k, k:]

    if M[n - 1, n - 1] == 0:
        raise ZeroDivisionError("Pivô nulo no último elemento: sistema singular.")

    # 3) Substituição regressiva
    x = np.zeros(n)
    for i in range(n - 1, -1, -1):
        x[i] = (M[i, n] - M[i, i + 1:n] @ x[i + 1:]) / M[i, i]
    return x

In [4]:
# Teste com o sistema do exercício 2
A = [[ 2,  1, -1],
     [-3, -1,  2],
     [-2,  1,  2]]
b = [8, -11, -3]

x_gauss = gauss_sem_pivoteamento(A, b)
x_numpy = np.linalg.solve(A, b)

print("Gauss (implementado):", x_gauss)
print("numpy.linalg.solve  :", x_numpy)
print("Diferença máxima    :", np.max(np.abs(x_gauss - x_numpy)))

Gauss (implementado): [ 2.  3. -1.]
numpy.linalg.solve  : [ 2.  3. -1.]
Diferença máxima    : 5.551115123125783e-16


**Interpretação:** o algoritmo reproduz a solução manual $(2,3,-1)$ e concorda com `numpy.linalg.solve`, que usa fatoração LU com pivoteamento parcial. Aqui os pivôs ($2$, $1/2$, $-1$) são não nulos, então a versão sem pivoteamento funciona. Em geral, sem pivoteamento o método falha se algum pivô for zero e pode perder precisão se for muito pequeno.

## Parte 4 — Aplicação

### Exercício 5 — Modelo de balanço térmico

$$\begin{bmatrix}4&-1&0\\-1&4&-1\\0&-1&3\end{bmatrix}\begin{bmatrix}T_1\\T_2\\T_3\end{bmatrix}=\begin{bmatrix}15\\10\\10\end{bmatrix}$$

#### a) Resolução manual

$$\left[\begin{array}{ccc|c}4&-1&0&15\\-1&4&-1&10\\0&-1&3&10\end{array}\right]$$

**Etapa 1.** $L_2\leftarrow L_2+\tfrac14 L_1$:
$$\left[\begin{array}{ccc|c}4&-1&0&15\\0&15/4&-1&55/4\\0&-1&3&10\end{array}\right]$$

**Etapa 2.** $L_3\leftarrow L_3+\tfrac{4}{15}L_2$:
$$\left[\begin{array}{ccc|c}4&-1&0&15\\0&15/4&-1&55/4\\0&0&41/15&41/3\end{array}\right]$$

**Substituição regressiva:**
- $\tfrac{41}{15}T_3=\tfrac{41}{3}\Rightarrow T_3=5$
- $\tfrac{15}{4}T_2-5=\tfrac{55}{4}\Rightarrow T_2=5$
- $4T_1-5=15\Rightarrow T_1=5$

**Solução:** $T_1=T_2=T_3=5$.

#### b) Implementação em Python

In [5]:
# Usa a função gauss_sem_pivoteamento definida no Exercício 4
A = [[ 4, -1,  0],
     [-1,  4, -1],
     [ 0, -1,  3]]
b = [15, 10, 10]

T_gauss = gauss_sem_pivoteamento(A, b)
T_numpy = np.linalg.solve(A, b)

print("T (Gauss)  :", T_gauss)
print("T (numpy)  :", T_numpy)
print("Resíduo ||AT - b||:", np.linalg.norm(np.array(A) @ T_gauss - b))

T (Gauss)  : [5. 5. 5.]
T (numpy)  : [5. 5. 5.]
Resíduo ||AT - b||: 0.0


**Interpretação:** a matriz é tridiagonal e diagonalmente dominante, o que garante pivôs não nulos e boa estabilidade sem pivoteamento. Fisicamente, as três temperaturas ficam em equilíbrio em 5 unidades, e o resultado confere com a solução manual.